In [ ]:
from bs4 import BeautifulSoup
import requests
import re
import json
import os
import urllib.request
import shutil
import math
import pandas as pd

#######################################
#         File EXTRACTOOOOOOOR        #
#######################################

# ASSUMPTIONS
# Input - URL and Domain provided
# File URL contains any of the keywords in "knownFileKeywords"
# No custom file path required in generated schema (default "/files/document.pdf")

# COMMON ERRORS/ISSUES
# If files are not detected, check if a keyword exist in "knownFileKeywords", add if missing
# If certain files are not downloaded, make sure file type is whitelisted in "whitelistedFileType", add if missing

# Define target website
URL = "https://www.mccy.gov.sg/sector/co-ops"
# Define the domain
domain = "https://www.mccy.gov.sg"

# Define variables
files = {}
folder_name = "docs"
hrefSchema = {}
whitelistedFileType = ["pdf", "xls", "xlsx", "csv", "tsv"]

# Add keyword to recognise file URLs
knownFileKeywords = ["/docs/", "/media/"]

# Define custom file path in schema after "/files"
# e.g. docsFolderPath = "/folder1/folder2"
# Output -> "/files/folder1/folder2/document.pdf"
# Leave blank if no customisation needed
docsFolderPath = ""

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
    'Connection': 'keep-alive',
}

# Remove existing folders and/or files if exists from previous session
try:
    while os.path.exists(folder_name) is True:
        shutil.rmtree(folder_name)
except:
  pass

# Download the files based on the constructed dictionary
def downloadFiles():
  # Create the output folder if it doesn't exist
  if not os.path.exists(folder_name):
      os.makedirs(folder_name)

  # Configure request headers for downloading
  headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
    'Connection': 'keep-alive',
  }
  opener = urllib.request.build_opener()
  opener.addheaders = [(key, value) for key, value in headers.items()]
  urllib.request.install_opener(opener)

  # Iterate through each file and download it
  for index, value in enumerate(files):
    try:
        # Stores each dictionary item's download link
        url = files[index]['Download Link']

        # Define file path and name
        file_path = os.path.join(f"{folder_name}", f"{files[index]['File name']}")
        
        # Download file
        urllib.request.urlretrieve(url, file_path)

    except:
        print("Error:", url)

  # Generate file size metadata and print the JSON schema
  for index, value in enumerate(files):
    try:
        # Debating between dividing 1024 or 1000. Generally 1 Kilo = 1000(grams), But in Binary 1 Kilo = 1024 bytes
        fileSize = str(math.ceil(os.path.getsize(f"docs/{files[index]['File name']}")/1024))
        # Calculate and define file size
        fileSize = fileSize + ' KB' if len(fileSize) <= 3 else str(round((int(fileSize)/1000), 2)) + ' MB' if len(fileSize) >= 4 and len(fileSize) < 7 else str(fileSize) + ' B'    
        
        hrefSchema = {
              "type": "text",
              "marks": [
                {
                  "type": "link",
                  "attrs": {
                    "href": f"/files{docsFolderPath if True else None}/{files[index]['File name']}"
                    }
                }
              ],
              # Original File Name is used as we want to display Circular 1 instead of circular-1
              "text": f"{files[index]['Title']} [{'DOCX' if '.docx' in files[index]['Download Link'] else 'DOC' if '.doc' in files[index]['Download Link'] else 'XLXS' if '.xlxs' in files[index]['Download Link'] else 'XLS' if '.xls' in files[index]['Download Link'] else 'ZIP' if '.zip' in files[index]['Download Link'] else 'PDF'}, {fileSize}]"
            }
        print(json.dumps(hrefSchema))
    except Exception as e:
      pass

# Generate dictionary
def getDictionary(link):
  count = 0
  URL = f"{link}"

  # Pulls entire HTML code and parse it through BeautifulSoup
  page = requests.get(URL, headers=headers)
  soup = BeautifulSoup(page.content, "html.parser")

  # Loop through all anchor tags to find downloadable files
  for i in soup.find_all(['a']):
    href = i.get('href')
    # Check if the link is a document link
    try:
        for keyword in knownFileKeywords:
            if keyword in i['href']:
                # Build dictionary
                files[count] = {"Title": i.text, "File name": i['href'][:i['href'].find('?')].split("/")[-1], "File path": f"/files{docsFolderPath if True else None}/{i['href'].split("/")[-1] if '?' not in i['href'] else i['href'][:i['href'].find('?')].split("/")[-1]}", "Download Link": i['href'] if 'go.gov.sg' in i['href'] else domain + i['href'] if domain not in i['href'] else i['href']}
                r = requests.head(files[count]["Download Link"], allow_redirects=True)
                # Resolve potential redirects
                files[count]["Download Link"] = r.url.split('%')[0]
                files[count]["File name"] = r.url[:r.url.find('?')].split("/")[-1] if '?' in r.url else r.url.split("/")[-1]
                files[count]["File name"] = files[count]["File name"].split('%')[0]
                
                print("Downloading", files[count]["File name"], files[count]["Download Link"])
                count += 1
    
    except:
        print("Error:", files[count]["Download Link"])
    
  downloadFiles()

getDictionary(URL)

# Export report to .csv
df = pd.DataFrame.from_dict(files, orient='index')
df.to_csv("file-extractor-report.csv", index=False)
print("Report generated")

In [ ]:
from bs4 import BeautifulSoup
import requests
import re
import json
import os
import urllib.request
import shutil
import math
import pandas as pd

#######################################
#         Image EXTRACTOOOOOOOR       #
#######################################

# ASSUMPTIONS
# Input - URL and Domain provided
# <img> tags are used for images
# No custom image path required in generated schema (default "/image/butterfly.jpg")

# Define target website
URL = "https://www.worldcitiessummit.com.sg/partners-n-media/wcs-2024-sponsors"
# Define the domain
domain = "https://www.worldcitiessummit.com.sg"

# Define variables
files = {}
folder_name = "images"
hrefSchema = {}

# Define custom image path in schema after "/images"
# e.g. docsFolderPath = "/folder1/folder2"
# Output -> "/images/folder1/folder2/butterfly.jpg"
# Leave blank if no customisation needed
docsFolderPath = ""

# Remove existing folders and/or files if exists from previous session
try:
    while os.path.exists(folder_name) is True:
        shutil.rmtree(folder_name)
except:
  pass

# Download the images based on the constructed dictionary
def downloadImages():
  # Create the output folder if it doesn't exist
  if not os.path.exists(folder_name):
      os.makedirs(folder_name)

  # Configure request headers for downloading
  headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
    'Connection': 'keep-alive',
  }
  opener = urllib.request.build_opener()
  opener.addheaders = [(key, value) for key, value in headers.items()]
  urllib.request.install_opener(opener)

  # Iterate through each file and download it
  for index, value in enumerate(files):
    try:
        # Stores each dictionary item's download link
        url = files[index]['Download link']

        # Define file path and name
        file_path = os.path.join(f"{folder_name}", f"{files[index]['Updated image name']}")
        
        # Download image
        response = requests.get(url, headers=headers, stream=True)
        if response.status_code == 200:
            with open(file_path, "wb") as f:
                for chunk in response.iter_content(1024):
                    f.write(chunk)
        
        hrefSchema = {
            "type": "image",
            "src": f"/images{docsFolderPath if True else None}/{files[index]['Updated image name']}",
            "alt": ""
        }
        print(json.dumps(hrefSchema))
    except Exception as e:
        pass

def getDictionary():
  # Pulls entire HTML code and parse it through BeautifulSoup
  page = requests.get(URL)
  # Paste HTML of a embedded gallery if needed, else ignore
  # page = """<div class="fancybox-thumbs"><ul><li data-index="0" tabindex="0" class="fancybox-thumbs-active"><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/1.jpg" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/1.jpg" style="width: 133px; height: 75px; margin-top: 0px; margin-left: -15px;"></li><li data-index="1" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/17.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/17.png" style="width: 165px; height: 75px; margin-top: 0px; margin-left: -31px;"></li><li data-index="2" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/7.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/7.png" style="width: 165px; height: 75px; margin-top: 0px; margin-left: -31px;"></li><li data-index="3" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/5.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/5.png" style="width: 165px; height: 75px; margin-top: 0px; margin-left: -31px;"></li><li data-index="4" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/4.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/4.png" style="width: 165px; height: 75px; margin-top: 0px; margin-left: -31px;"></li><li data-index="5" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/27.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/27.png" style="width: 133px; height: 75px; margin-top: 0px; margin-left: -15px;"></li><li data-index="6" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/12.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/12.png" style="width: 165px; height: 75px; margin-top: 0px; margin-left: -31px;"></li><li data-index="7" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/26.jpg" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/26.jpg" style="width: 133px; height: 75px; margin-top: 0px; margin-left: -15px;"></li><li data-index="8" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/8.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/8.png" style="width: 165px; height: 75px; margin-top: 0px; margin-left: -31px;"></li><li data-index="9" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/6.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/6.png" style="width: 165px; height: 75px; margin-top: 0px; margin-left: -31px;"></li><li data-index="10" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/23.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/23.png" style="width: 166px; height: 75px; margin-top: 0px; margin-left: -31px;"></li><li data-index="11" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/29.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/29.png" style="width: 133px; height: 75px; margin-top: 0px; margin-left: -15px;"></li><li data-index="12" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/13.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/13.png" style="width: 164px; height: 75px; margin-top: 0px; margin-left: -30px;"></li><li data-index="13" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/20.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/20.png" style="width: 165px; height: 75px; margin-top: 0px; margin-left: -31px;"></li><li data-index="14" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/10.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/10.png" style="width: 165px; height: 75px; margin-top: 0px; margin-left: -31px;"></li><li data-index="15" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/11.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/11.png" style="width: 164px; height: 75px; margin-top: 0px; margin-left: -30px;"></li><li data-index="16" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/16.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/16.png" style="width: 164px; height: 75px; margin-top: 0px; margin-left: -30px;"></li><li data-index="17" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/3.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/3.png" style="width: 164px; height: 75px; margin-top: 0px; margin-left: -30px;"></li><li data-index="18" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/9.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/9.png" style="width: 166px; height: 75px; margin-top: 0px; margin-left: -31px;"></li><li data-index="19" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/25.jpg" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/25.jpg" style="width: 133px; height: 75px; margin-top: 0px; margin-left: -15px;"></li><li data-index="20" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/28.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/28.png" style="width: 133px; height: 75px; margin-top: 0px; margin-left: -15px;"></li><li data-index="21" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/19.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/19.png" style="width: 166px; height: 75px; margin-top: 0px; margin-left: -31px;"></li><li data-index="22" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/24.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/24.png" style="width: 133px; height: 75px; margin-top: 0px; margin-left: -15px;"></li><li data-index="23" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/21.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/21.png" style="width: 165px; height: 75px; margin-top: 0px; margin-left: -31px;"></li><li data-index="24" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/2.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/2.png" style="width: 165px; height: 75px; margin-top: 0px; margin-left: -31px;"></li><li data-index="25" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/15.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/15.png" style="width: 164px; height: 75px; margin-top: 0px; margin-left: -30px;"></li><li data-index="26" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/18.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/18.png" style="width: 166px; height: 75px; margin-top: 0px; margin-left: -31px;"></li><li data-index="27" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/14.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/14.png" style="width: 165px; height: 75px; margin-top: 0px; margin-left: -31px;"></li><li data-index="28" tabindex="0" class=""><img data-src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/22.png" src="https://www.geri.com.sg/Gallery/Media Gallery/Understanding Cognitive Decline in Older Adults - What’s on the Horizon/22.png" style="width: 165px; height: 75px; margin-top: 0px; margin-left: -31px;"></li></ul></div>"""
  soup = BeautifulSoup(page.content, "html.parser")

  # Loop through all anchor tags to find downloadable images
  for i, v in enumerate(soup.find_all(['img'])):
    try:    
        # Build dictionary
        files[i] = {"Image name": v['src'].split("/")[-1], "Image path": f"/images/{v['src'].split("/")[-1]}", "Download link": domain + v['src'].replace(" ", "%20") if "https" not in v['src'] else v['src']}
        print("Downloading", files[i]["Image name"], files[i]["Download link"])
        
    except Exception as e:
        print("Error:", files[i]["Download link"])

  downloadImages()

getDictionary()

# Export report to .csv
df = pd.DataFrame.from_dict(files, orient='index')
df.to_csv("image-extractor-report.csv", index=False)
print("Report generated")

In [ ]:
from bs4 import BeautifulSoup
import requests
import re
import json
import os
import urllib.request
import shutil
import math
import pandas as pd

#######################################
#     Link Extractooooor (Website)    #
#######################################

# ASSUMPTIONS
# Input - URL, Domain provided
# HTML div class is provided
# There is text in the agency's href HTML code (e.g. <a href="www.google.com>This is a text</a>)

# Define target website
url = "https://www.healthprofessionals.gov.sg/smc/announcements"
# Define domain
domain = "https://www.healthprofessionals.gov.sg"
files = {}

# Pulls entire HTML code and parse it through BeautifulSoup
page = requests.get(url)
soup = BeautifulSoup(page.content, "html.parser")
soup = soup.find("div", class_="col-sm-12 announcement-all")

# Loop through all anchor tags to find links
for i, v in enumerate(soup.find_all(['a'])):
    print("Title:", v.text)
    # Build dictionary for easier search
    files[i] = {"Title": v.text, "Link": domain + v['href']}
    
# Export report to .csv
df = pd.DataFrame.from_dict(files, orient='index')
df.to_csv("links-report.csv", index=False)
print("Report generated")

In [ ]:
import pandas as pd
import json
import os
import urllib.request
import shutil
import requests

########################################
#     Collection Generator (Link)      #
########################################

# ASSUMPTIONS
# Title - English
# Input - CSV file provided

# CSV REQUIREMENTS (Use the below as the column name)
# 1. Page name
# 2. Link
# 4. Date (Optional, remove comment in schema if used)
# 4. Category

# COMMON ERRORS/ISSUES
# 1. Generated JSON file name contains alot of "-" (e.g. ----.json) -> Full of invalid characters in title (e.g. chinese) 

# Define dataset
df = pd.read_csv('in-the-press-links.csv') # <--- Enter CSV file name here
df_copy = df.copy()

# Define the folder name
file_name = 'json'

whitelistCharacters = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '-']

duplicate = {}

# Remove existing folders and/or files if exists from previous session
try:
    while os.path.exists(file_name) or os.path.exists(file_name_zip) is True:
        shutil.rmtree(file_name)
except:
  pass

def generateSchema():
    # Clean title to use as JSON file name
    for index, value in enumerate(df_copy['Page name']):
        temp = ""
        for char in df_copy["Page name"][index]:
            if char.lower() not in whitelistCharacters:
                temp += "-"
            else:
                temp += char.lower()
        df_copy.loc[index, 'renamed'] = temp

        # Generate schema
        try:
            data = {
              "version": "0.1.0",
              "layout": "link",
              "page": {
                "title": f"{df_copy['Page name'][index]}",
                "ref": f"{df_copy['Link'][index]}",
                "category": f"{df_copy['Category'][index]}",
                # Comment out Date if unused, vice-versa
                "date": f"{df_copy["Date"][index]}"
              },
              "content": []
            }
        except:
            print("Error:", df_copy['Page name'][index])

        # Create the directory if it doesn't exist
        os.makedirs(file_name, exist_ok=True)

        # Create the file path using os.path.join
        file_path = os.path.join(file_name, f"{df_copy['renamed'][index]}.json")

        if df_copy['renamed'][index] not in duplicate:
            duplicate[df_copy['renamed'][index]] = 0
        
        # Add index to name to prevent duplication
        if os.path.exists(file_path):   
            duplicate[df_copy['renamed'][index]] += 1
            file_path = os.path.join(file_name, f"{df_copy['renamed'][index]}-{duplicate[df_copy['renamed'][index]]}.json")

        # Create json file
        with open(f"{file_path}", 'w+', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=4)
            print("Generating:", df_copy['Page name'][index])

generateSchema()

In [ ]:
import pandas as pd
import json
import os
import urllib.request
import shutil
import requests
import math

########################################
#     Collection Generator (Files)     #
########################################

# ASSUMPTIONS
# 1. Input - CSV file provided
# 2. Link is using file links (https:www.) instead of file path (/files/folder/document.pdf)
# 2.1 Easier to calculate file size
# 2.2 More work (Title, Category, Date) when preparing CSV if use existing files
# 2.3 Can use Link extractor to compile file links

# CSV REQUIREMENTS (Use the below as the column name)
# 1. Page title
# 2. Link
# 3. Category
# 4. Date (Optional, remove comment in schema if used)

# Define dataset
df = pd.read_csv('NParks.csv') # <--- Enter CSV file name here
df_copy = df.copy()

# Define the folder path
file_name = 'docs'
json_name = 'json'

whitelistCharacters = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '-', '.']

# Remove existing folders and/or files if exists from previous session
try:
    while os.path.exists(file_name) is True:
        shutil.rmtree(file_name)
    while os.path.exists(json_name) is True:
        shutil.rmtree(json_name)
except:
  pass

# Split the folder structure to get the file name via the last element
def processCSV():
    # Clean title to use as JSON file name
    for index, value in enumerate(df_copy['Page title']):
        temp = ""
        for name in value:
            if name.lower() not in whitelistCharacters:
                temp+="-"
            else:
                temp+=name.lower()
        df_copy.loc[index, 'renamed'] = temp
    downloadFiles()
    generateSchema()

# Generate schema
def generateSchema():
    for index, value in enumerate(df_copy['Page title']):
        data = {
            "version": "0.1.0",
            "layout": "link",
            "page": {
                "title": df_copy["Updated title"][index],
                # File path can be customised for convenience
                "ref": f"/files/{df_copy['renamed'][index]}{'.xlsx' if '.xlsx' in df_copy['Download link'][index] else '.xls' if '.xls' in df_copy['Download link'][index] else '.pdf'}",
                # Comment out Date if unused, vice-versa
                # "date": df_copy["Date"][index],
                "category": df_copy["Category"][index]
            },
            "content": []
        }

        # Create the directory if it doesn't exist
        os.makedirs(json_name, exist_ok=True)

        # Create the file path using os.path.join
        file_path = os.path.join(json_name, f"{df_copy['renamed'][index]}.json")

        # Create json file
        with open(f"{file_path}", 'w+', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=4)
            print("Generating", f"{df_copy['renamed'][index].split('.')[0]}.json")

# Download files
def downloadFiles():
    for index, value in enumerate(df_copy['Page title']):
        try:
            # Check if Download link is invalid
            page = requests.get(df_copy['Download link'][index])
            if page.status_code != 200:
                print(df_copy['renamed'][index])
                
            # Create the directory if it doesn't exist
            os.makedirs(file_name, exist_ok=True)
            
            # Create the file path using os.path.join
            file_path = os.path.join(file_name, f"{df_copy['renamed'][index]}{'.xlsx' if '.xlsx' in df_copy['Download link'][index] else '.xls' if '.xls' in df_copy['Download link'][index] else '.pdf'}")
            
            # Download file from URL
            urllib.request.urlretrieve(df_copy['Download link'][index], file_path)
    
            print("Downloading", df_copy['renamed'][index])
            
            # Debating between dividing 1024 or 1000. Generally 1 Kilo = 1000(grams), But in Binary 1 Kilo = 1024 bytes
            fileSize = str(math.ceil(os.path.getsize(f"docs/{df_copy['renamed'][index]}{'.docx' if '.docx' in df_copy['Download link'][index] else '.doc' if '.doc' in df_copy['Download link'][index] else '.xlxs' if '.xlxs' in df_copy['Download link'][index] else '.xls' if '.xls' in df_copy['Download link'][index] else '.zip' if '.zip' in df_copy['Download link'][index] else '.pdf'}")/1024))
            # Calculate and define file size
            fileSize = fileSize + ' KB' if len(fileSize) <= 3 else str(round((int(fileSize)/1000), 2)) + ' MB' if len(fileSize) >= 4 and len(fileSize) < 7 else str(fileSize) + ' B'
    
            df_copy.loc[index, "Updated title"] = df_copy["Page title"][index] + " [" + fileSize + "]"
        except:
            print(df_copy["Page title"][index], "File not found!")

processCSV()

In [ ]:
import pandas as pd
import json
import os
import urllib.request
import shutil
import requests
from bs4 import BeautifulSoup

########################################
#     HTML Extractoooor (URL, SMC)     #
########################################

# ASSUMPTION
# Input - Website is provided
# Target div class is provided

# Define target website
URL = "https://www.healthprofessionals.gov.sg/smc/feedback"

pageData = {}

# Configure request headers for downloading
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
    'Connection': 'keep-alive',
}

# Send a GET request to fetch the page HTML
page = requests.get(URL, headers=headers)
soup = BeautifulSoup(page.content, "html.parser")

# Find all <div> tag with the class "panel-body"
questions = soup.find_all("h3", class_="accordion")
for i, v in enumerate(questions):
    pageData[i] = {"Questions": v.get_text()}
    print("Extracted:", v.get_text())

answers = soup.find_all("div", class_="show-more")
for i, v in enumerate(answers):
    pageData[i]["Answers"] = v
    print("Extracted:", v.get_text())

# Generate report in .csv
df = pd.DataFrame.from_dict(pageData, orient='index')
df.to_csv("QnA.csv", index="false",)
print("Report generated")

In [ ]:
import pandas as pd
import json
import os
import urllib.request
import shutil
import requests
from bs4 import BeautifulSoup

########################################
#     HTML Extractoooor (CSV, REACH)   #
########################################

# ASSUMPTIONS
# Title - English
# Input - CSV file provided

# CSV REQUIREMENTS (Use the below as the column name)
# 1. Page name
# 2. Link

# Define dataset
df = pd.read_csv('reach-automation.csv') # <--- Enter CSV file name here
df_copy = df.copy()

pageData = []

# Loop through each URL in the csv file
for index, value in enumerate(df_copy['Link']):
    # Fetch site's HTML
    page = requests.get(df_copy['Link'][index])
    soup = BeautifulSoup(page.content, "html.parser")
    
    # Extract the content from "div" container and class
    content = soup.find_all("div", class_="sfContentBlock sf-Long-text a-rich-text")
    pageData.append(content)

# Insert extracted list of HTML to a new column called "HTML"
df_copy["HTML"] = pageData

# Export to CSV
df_copy.to_csv("automation-processed.csv", index="false",)
print("Report generated")

In [ ]:
from pathlib import Path
import json
import pandas as pd
import os

########################################
#          JSON Updater (ACE)          #
########################################

# ASSUMPTIONS
# 1. Mass overwritting is required to a folder of JSON files (e.g. Changing of all Categories / Taggings)
# 2. All existing edits to site repo have been pushed
# 2.1 Painful to undo changes done by the script if there are existing changes not pushed
# 3. Folder path is provided
# 4. Only Category is to be changed in this script
# 5. CSV's title matches JSON file's title

# CSV Requirements
# 1. Title
# 2. Category

# Define dataset
df = pd.read_csv('ace-med-tech-1.csv')
df_copy = df.copy()

pageData = {}
whitelistCharacters = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '-']

# List all JSON files in this directory
file_paths = os.listdir(f"/Users/yongteng/Documents/GitHub/moh-ace-hta-next/schema/healthcare-professionals/ace-technology-guidances/medical-technology-guidance")

for index, value in enumerate(df_copy["Title"]):
    # Using "Title" as the key to store the category
    pageData[value] = {"category": df_copy["Category"][index]}

# Loop through each file
for i in file_paths:
    file_path = f"/Users/yongteng/Documents/GitHub/moh-ace-hta-next/schema/healthcare-professionals/ace-technology-guidances/medical-technology-guidance/{i}"
    # Ignore sys file
    if i != ".DS_Store":
        with Path(file_path).open("r", encoding="utf-8") as f:
            # Read JSON file
            content = f.read()
            # Load file in JSON format
            page = json.loads(content)

            # Using JSON file's title to retrieve and overwrite "category"
            page["page"]["category"] = pageData[page["page"]["title"]]["category"]
            
            # Save changes made to the JSON file
            with open(file_path, 'w') as file:
                json.dump(page, file, indent=2)

In [ ]:
import requests
from requests.auth import HTTPBasicAuth
from urllib.parse import urljoin, urlparse, urlunparse
import json
from bs4 import BeautifulSoup
import ssl
import xml.etree.ElementTree as ET
from pathlib import Path
import pandas as pd
from requests.exceptions import SSLError, ConnectionError, RequestException, Timeout

########################################
#          Broken link checker         #
########################################

# ASSUMPTIONS
# Staging site provided
# Password provided
# Folder path to repo provided

# Define folder path to repo
# This is to whitelist navbar and footer links to reduce checks
folder_path = "/Users/yongteng/Documents/GitHub/mccy-corp-next/"

# Define password
password = ""

# Define staging site
domain = ""
url = domain + "/sitemap.xml"

# Get staging site's sitemap
response = requests.get(url, auth=HTTPBasicAuth('user', password))

# Parse the XML content
root = ET.fromstring(response.text)

# Define the namespace to use when accessing the XML tags
namespaces = {'': 'http://www.sitemaps.org/schemas/sitemap/0.9'}

# Store all page paths extract from sitemap.xml
links = []

# Store all links and href that the script deemed inaccessible
# Warning: Will include false positives
errorPages = {}

# Store a list of whitelisted (common) sites the script will automatically skip (e.g. gov.sg/trusted-sites)
# Avoid unintentionally getting flagged by monitoring tools
whitelistedSites = set()

# List of error codes
errorCodes = { 
    400: "Bad Request",
    401: "Unauthorized",
    404: "Not found",
    410: "Gone",
    403: "Forbidden",
    500: "Internal Server Error",
    502: "Bad gateway",
    503: "Service unavailable",
    504: "Gateway Timeout",
    505: "HTTP Version Not Supported"
  }

# Find all 'url' elements in sitemap.xml
for elem in root.findall('.//url', namespaces):  
    # Find 'loc' inside 'url' in sitemap.xml
    loc = elem.find('loc', namespaces)
    if loc is not None:
        links.append(loc.text.replace("https://www.isomer.gov.sg", domain))

# Clean up fragment, query, and trailing slash in url (e.g. /, /#anchor, /#)
def normalize_url(url):
    parsed = urlparse(url)
    normalized = parsed._replace(fragment='', query='', path=parsed.path.rstrip('/'))
    return urlunparse(normalized)

# Get navbar and footer paths to whitelist, easier than scrapping from staging site
def getNavFooterLinks():
    navbar_path = folder_path + "data/navbar.json"
    footer_path = folder_path + "data/footer.json"
    # Whitelist common links found in all pages
    whitelistedSites.update([domain, "https://www.gov.sg/trusted-sites#govsites", "https://www.gov.sg/trusted-sites", "https://go.gov.sg/report-vulnerability", "https://www.reach.gov.sg", "https://www.isomer.gov.sg", "https://www.open.gov.sg"])

    # Navbar
    with Path(navbar_path).open("r", encoding="utf-8") as f:
        # Read JSON file
        navbar = json.loads(f.read())
        for i, v in enumerate(navbar):
            try:
                for x in range(0, len(v["items"])):
                    path = normalize_url(v["items"][x]["url"])
                    whitelistedSites.add(domain + path if "https" not in path else path)
            except:
                path = normalize_url(v["url"])
                whitelistedSites.add(domain + path if "https" not in path else path)
    
    # Footer
    with Path(footer_path).open("r", encoding="utf-8") as f:
        # Read JSON file
        footer = json.loads(f.read())
        for i, v in enumerate(footer):
            try:
                for x in range(0, len(footer[v])):
                    path = normalize_url(footer[v][x]["url"])
                    whitelistedSites.add(domain + path if "https" not in path else path)
            except:
                path = normalize_url(footer[v])
                whitelistedSites.add(domain + path if "https" not in path else path)

# Check external link's response code
# Logged if response code is not 200
def checkExternalLink(parenturl, url):
    # print(parenturl, url)
    try:
        resp = requests.get(url, allow_redirects=True)
        # print("Checking external:", url, "Response code:", resp.status_code)
        if resp.status_code != 200:
            if resp.status_code in errorCodes:
                errorPages[len(errorPages)] = {"URL": parenturl, "href": url, "status code": resp.status_code}
                print("URL", parenturl, "href", url, "Status code", f"{resp.status_code} - {errorCodes[resp.status_code]}")
 
    except RequestException as e:
        errorPages[len(errorPages)] = {"URL": parenturl, "href": url}
        print("URL", parenturl, "href", url, "Exception", e)

# Check if internal link redirects to domain/404.html (broken link)
def checkInternalLink(parenturl, url):
    # print(parenturl, url)
    try:
        page = requests.head(url, auth=HTTPBasicAuth('user', password), allow_redirects=True)
        # print("Checking internal:", url, "Response code:", page.status_code)
        # Check if it redirects to a 404 page
        if page.url == domain + "/404.html":
            errorPages[len(errorPages)] = {"URL": parenturl, "href": url}
            print("URL", parenturl, "href", url)
            
    except RequestException as e:
        errorPages[len(errorPages)] = {"URL": parenturl, "href": url}
        print("URL", parenturl, "href", url, "Exception", e)

# Extract all links in the URL
# Add staging site domain if its internal
# Return a list of links to check for broken links
def getAllLinks(url):
    response = requests.get(url, auth=HTTPBasicAuth('user', password))
    soup = BeautifulSoup(response.content, "html.parser")
    href = set()
    for a in soup.find_all("a"):
        temp = a.get('href')
        if "mailto" not in temp:
            link = normalize_url(domain + temp if "http" not in temp else temp)
            if link not in whitelistedSites:
                href.add(link)
    return href

# Start building a list of whitelisted sites for the broken link checker to skip
getNavFooterLinks()

# 1. Loop through each page (we call this the parent page) found in sitemap.xml
# 2. Check if the parent page is broken or not
# 3. Retrieve all links fonud in parent page
# 4. Check status of each link (broken or not)
for i, v in enumerate(links):
    # print(v)
    checkInternalLink(v, v) if "amplifyapp" in v else checkExternalLink(v, v)
    # TO DO: Skip checking for broken links if external link
    for index, link in enumerate(getAllLinks(v)):
        checkInternalLink(v, link) if "amplifyapp" in link else checkExternalLink(v, link)

(pd.DataFrame.from_dict(data=errorPages, orient='index')
   .to_csv('ErrorPages.csv', header=True))